# C6-pytorch — Practice p14 — Solution


`ScaledDense` registers all three stored vectors/matrices uniformly as
frozen parameters.  The audit then derives every result from module
inspection rather than duplicating the expected values by hand.


In [ ]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)

class ScaledDense(nn.Module):
    def __init__(self, weight, bias, scale):
        super().__init__()
        self.weight = nn.Parameter(torch.as_tensor(weight), requires_grad=False)
        self.bias = nn.Parameter(torch.as_tensor(bias), requires_grad=False)
        self.scale = nn.Parameter(torch.as_tensor(scale), requires_grad=False)

    def forward(self, x):
        return self.scale * (x @ self.weight.T + self.bias)


class LooseDense(nn.Module):
    """A teammate's rewrite of ScaledDense. Same arithmetic. Audit the storage."""

    def __init__(self, weight, bias, scale):
        super().__init__()
        self.weight = torch.as_tensor(weight)
        self.bias = nn.Parameter(torch.as_tensor(bias))
        self.scale = nn.Parameter(torch.as_tensor(scale), requires_grad=False)

    def forward(self, x):
        return self.scale * (x @ self.weight.T + self.bias)


Wg = torch.tensor([[1.0, 0.0, -1.0], [2.0, 1.0, 0.0]])
bg = torch.tensor([0.5, -1.0])
sg = torch.tensor([2.0, 0.5])
x14 = torch.tensor([[1.0, 2.0, 3.0], [0.0, 1.0, 0.0],
                    [-1.0, 0.5, 2.0], [4.0, -2.0, 1.0]])

good = ScaledDense(Wg, bg, sg)
y_good = good(x14)
n_good = len(list(good.parameters()))
good_frozen = all(not p.requires_grad for p in good.parameters())

loose = LooseDense(Wg, bg, sg)
n_loose = len(list(loose.parameters()))
loose_flags = sorted(p.requires_grad for p in loose.parameters())
missing_key = next(iter(set(good.state_dict()) - set(loose.state_dict())))
same_out = bool((good(x14) == loose(x14)).all())

y_good, n_good, good_frozen, n_loose, loose_flags, missing_key, same_out


`weight` is an unregistered plain tensor, exposed by the missing
`weight` entry in `state_dict()`, while `bias` is registered but left
with `requires_grad=True`, exposed by `parameters()` and its flags.
Identical outputs make both defects dangerous because a forward-only
smoke test passes while saving/inspection silently loses a weight and
misstates the intended frozen status.


### Answer check


In [ ]:
expected_y = torch.tensor([[-3.0, 1.5], [1.0, 0.0], [-5.0, -1.25], [7.0, 2.5]])
assert torch.equal(y_good, expected_y)
assert n_good == 3 and good_frozen
assert n_loose == 2 and loose_flags == [False, True]
assert missing_key == "weight"
assert same_out is True
